## RAG Pipelines - Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [2]:
### Read all the PDf's inside the directory

def process_all_pdfs(pdf_directory):
    """Process all the pdf files in a directory."""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # find all PDF files recursively 
    pdf_files  = list(pdf_dir.glob("**/*.pdf"))
    print(f"found {len(pdf_files)} PDF files to process.")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            ## add source information to metadata
            for doc in documents:
                doc.metadata['sourece_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)    
            print(f"Loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents     


# process all pdfs in the data directory
all_pdf_documents = process_all_pdfs("../data")

found 4 PDF files to process.

Processing: Course_.pdf
Loaded 3 pages

Processing: Discussion Summary_Srijan_.pdf
Loaded 17 pages

Processing: MLOPS.pdf
Loaded 326 pages

Processing: Research Paper Type 1.pdf
Loaded 3 pages

Total documents loaded: 349


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m146 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Course_', 'source': '..\\data\\pdf\\Course_.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'sourece_file': 'Course_.pdf', 'file_type': 'pdf'}, page_content='NAME  -  Srijan   Course  -  Ethical  Decision  Making  For  Success  in  the  Tech             \nIndustry\n  Course  No   -  OC30001   Final  Course  Reflection  -   \nIn  today’s  highly  competitive  technology  environment,  \norganizations\n \nfrequently\n \nencounter\n \nsituations\n \nwhere\n \nbusiness\n \npressure\n \nconflicts\n \nwith\n \nethical\n \nresponsibility.\n \nOne\n \nsignificant\n \nexample\n \nis\n \nthe\n \nchoice\n \nbetween\n \nreleasing\n \na\n \nproduct\n \nquickly\n \nto\n \ncapture\n \nmarket\n \nattention\n \nor\n \npostponing\n \nthe\n \nlaunch\n \nto\n \nensure\n \nthat\n \nthe\n \nsystem\n \nis\n \nsecure,\n \nstable,\n \nand\n \nreliable.\n \nWhile\n \nan\n \nearly\n \nrel

In [4]:
## Text splitting into chunks 

def split_documents(documents,chunk_size = 1000 , chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n","\n"," ",""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs    

In [5]:
chunks = split_documents(all_pdf_documents)
chunks

Split 349 documents into 736 chunks

Example chunk:
Content: NAME  -  Srijan   Course  -  Ethical  Decision  Making  For  Success  in  the  Tech             
Industry
  Course  No   -  OC30001   Final  Course  Reflection  -   
In  today’s  highly  competitive  ...
Metadata: {'producer': 'Skia/PDF m146 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Course_', 'source': '..\\data\\pdf\\Course_.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'sourece_file': 'Course_.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Skia/PDF m146 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Course_', 'source': '..\\data\\pdf\\Course_.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'sourece_file': 'Course_.pdf', 'file_type': 'pdf'}, page_content='NAME  -  Srijan   Course  -  Ethical  Decision  Making  For  Success  in  the  Tech             \nIndustry\n  Course  No   -  OC30001   Final  Course  Reflection  -   \nIn  today’s  highly  competitive  technology  environment,  \norganizations\n \nfrequently\n \nencounter\n \nsituations\n \nwhere\n \nbusiness\n \npressure\n \nconflicts\n \nwith\n \nethical\n \nresponsibility.\n \nOne\n \nsignificant\n \nexample\n \nis\n \nthe\n \nchoice\n \nbetween\n \nreleasing\n \na\n \nproduct\n \nquickly\n \nto\n \ncapture\n \nmarket\n \nattention\n \nor\n \npostponing\n \nthe\n \nlaunch\n \nto\n \nensure\n \nthat\n \nthe\n \nsystem\n \nis\n \nsecure,\n \nstable,\n \nand\n \nreliable.\n \nWhile\n \nan\n \nearly\n \nrel

## Embedding and vectorStore DB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List , Dict , Any , Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


c:\Users\KIIT0001\Desktop\Agentic_AI\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\KIIT0001\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8682.81it/s]


Model loaded successfully. Embedding dimension: 384


## VectorStore

In [8]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [10]:
## convert the text to embeddings 

texts = [doc.page_content for doc in chunks]

## Generate the Embeddings 

embeddings  = embedding_manager.generate_embeddings(texts)

## store in the vector database
vectorstore.add_documents(chunks,embeddings)


Generating embeddings for 736 texts...


Batches:   0%|          | 0/23 [00:00<?, ?it/s]

Batches: 100%|██████████| 23/23 [00:26<00:00,  1.14s/it]


Generated embeddings with shape: (736, 384)
Adding 736 documents to vector store...
Successfully added 736 documents to vector store
Total documents in collection: 736


## Retriever Pipeline From VectorStore

In [11]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [12]:
RAGRetriever    

__main__.RAGRetriever

In [13]:
rag_retriever.retrieve("What is MLOPS ?")

Retrieving documents for query: 'What is MLOPS ?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 50.17it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


[]

In [14]:
rag_retriever.retrieve("Emotion recognition using psychological signals")

Retrieving documents for query: 'Emotion recognition using psychological signals'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 64.90it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_752f4850_724',
  'content': 'accuracies of 76.82%, 75.13%, 76.01%, and 77.34% across the\nfour dimensions, consistently outperforming standalone baseline\nmodels.\nIndex Terms—Affective Computing, EEG, Empirical Mode\nDecomposition, Ensemble Learning, Feature Selection, DEAP.\nI. INTRODUCTION\nHuman emotion plays a fundamental role in cognition,\nsocial interaction, and decision-making. In recent years, train-\ning computational systems to recognize and adapt to these\nemotional states—a field known as affective computing—has\ngained significant traction. Among the various physiological\nmarkers used to infer emotion, Electroencephalography (EEG)\nis highly valued as it captures neuro-electrical activity directly\nfrom the central nervous system, offering a direct window into\naffective states.\nHowever, extracting reliable emotional indicators from EEG\ndata is notoriously difficult. The signals are inherently non-\nstationary, highly susceptible to noise, and vary drasti

In [15]:
rag_retriever.retrieve("What is correlation removal ?")

Retrieving documents for query: 'What is correlation removal ?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 57.51it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


[]

In [16]:
rag_retriever.retrieve("Limitations of PCA and how LDA works ?")

Retrieving documents for query: 'Limitations of PCA and how LDA works ?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 39.46it/s]

Generated embeddings with shape: (1, 384)
Retrieved 4 documents (after filtering)


[{'id': 'doc_2cb14c54_628',
  'content': 'Dimensionality Reduction\n[ 258 ]\nThat\'s all. Note that in contrast to the previous PCA example, we provide the class \nlabels to the fit_transform() method. Thus, PCA is an unsupervised feature \nextraction method, whereas LDA is a supervised one. The result looks as expected:\nSo, why then consider PCA at all and not simply use LDA? Well, it\'s not that simple. \nWith an increasing number of classes and fewer samples per class, LDA does not \nlook that well any more. Also, PCA seems to be not as sensitive to different training \nsets as LDA. So, when we have to advise which method to use, we can only suggest a \nclear "it depends".\nMultidimensional scaling\nAlthough, PCA tries to use optimization for retained variance, multidimensional \nscaling (MDS) tries to retain the relative distances as much as possible when \nreducing the dimensions. This is useful when we have a high-dimensional  \ndataset and want to get a visual impression.',
  '

In [18]:
import os
from dotenv import load_dotenv
load_dotenv()



True

In [19]:
from langchain_groq import ChatGroq
from langchain.prompts import PromptTemplate
from langchain.schema import HumanMessage, SystemMessage

ModuleNotFoundError: No module named 'langchain.prompts'